In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
# import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
import os
import datetime
import pandas as pd

today = datetime.date.today().strftime("%Y-%m-%d")

output_filename = f"bindingdb_all_2026-07-27_"

In [2]:
df = pd.read_csv('data/processed/preprocessed_bindingdb.csv', nrows=20000, low_memory=False)

df.info()  # Check the structure of the DataFrame

print("First few rows of the DataFrame:")
print(df.head())


# 👉 If the column is named something else (e.g., 'score', 'value'), change this:
# affinity_vals = df['score'].dropna().values
affinity_vals = df['affinity'].dropna().values  # <-- most common case

# Optional: inspect first few values
print("Sample affinity values:", affinity_vals[:5])
print("Range:", affinity_vals.min(), "to", affinity_vals.max())

# 🔹 2. Plot histogram (matching your style)
plt.figure(figsize=(6, 4))
# Use bins that give clean, stepped bars (like your image)
n_bins = min(30, int(np.sqrt(len(affinity_vals))) + 5)
plt.hist(affinity_vals, 
         bins=n_bins, 
         color='#0077A0',      # teal blue
         edgecolor='none',     # no black outlines
         alpha=1.0)

# 🔹 3. Style to match your image:
plt.title('# affinity', fontsize=14, fontweight='bold', loc='left')
plt.xlabel('')
plt.ylabel('')
plt.xlim(0, affinity_vals.max())              
plt.xticks([0, affinity_vals.max()])
plt.yticks([])                 # hide y-axis ticks/labels
plt.tight_layout()
plt.show()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   smiles    20000 non-null  str    
 1   sequence  20000 non-null  str    
 2   affinity  20000 non-null  float64
dtypes: float64(1), str(2)
memory usage: 468.9 KB
First few rows of the DataFrame:
                                              smiles  \
0  *.C[C@H]1CO[C@@H](c2ccc(N)nc2)CN1C[C@@H]1C[C@H...   
1  *.C[C@H]1CO[C@@H](c2ccc(N)nc2)CN1C[C@@H]1C[C@H...   
2  *.Cc1ccc(F)c2c(=O)[nH]c(CCCN3CC[C@H](c4ccc(F)c...   
3  *C(C)(F)c1nc(CC)cc(Nc2cc(NC(C)=O)ncc2-c2cnc(OC...   
4  *C(F)(F)CN1CCC(C(=O)N[C@@H](Cc2cccc3c(-c4nccc(...   

                                            sequence  affinity  
0  MASLSQLSGHLNYTCGAENSTGASQARPHAYYALSYCALILAIVFG...  6.043676  
1  MDPLNLSWYDDDLERQNWSRPFNGSDGKADRPHYNYYATLLTLLIA...  5.115797  
2  MAESSDKLYRVEYAKSGRASCKKCSESIPKDSLRMAIMVQSPMFDG...  9.000000  
3  GETSNLIIMRGARASPRTLNLSQLS

NameError: name 'plt' is not defined

In [4]:

class DTICNN(nn.Module):
    def __init__(self, ligand_dim=1024, protein_dim=320, conv_channels=[64, 128], 
                 fc_dims=[512, 256, 1], dropout_rate=0.3):
        super(DTICNN, self).__init__()
        
        # Ligand processing branch - expects [batch, channels, length]
        self.ligand_conv = nn.Sequential(
            nn.Conv1d(1, conv_channels[0], kernel_size=5, padding=2),  # [1, 1024] -> [64, 1024]
            nn.BatchNorm1d(conv_channels[0]),
            nn.ReLU(),
            nn.MaxPool1d(2),                                           # [64, 1024] -> [64, 512]
            
            nn.Conv1d(conv_channels[0], conv_channels[1], kernel_size=3, padding=1),  # [64, 512] -> [128, 512]
            nn.BatchNorm1d(conv_channels[1]),
            nn.ReLU(),
            nn.MaxPool1d(2),                                           # [128, 512] -> [128, 256]
        )
        
        # For protein embeddings (320-dim), use fully connected layers instead of convolutions
        # since 320 is too small for multiple conv/maxpool operations
        self.protein_fc = nn.Sequential(
            nn.Linear(protein_dim, conv_channels[0]),
            nn.BatchNorm1d(conv_channels[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(conv_channels[0], conv_channels[1]),
            nn.BatchNorm1d(conv_channels[1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        
        # Calculate flattened dimensions after convolutions
        ligand_flat_dim = conv_channels[1] * 256  # 128 * 256 = 32768 (after conv + pooling)
        protein_flat_dim = conv_channels[1]       # 128 (after FC layers)
        
        # Combined fully connected layers
        self.fc_layers = nn.Sequential(
            nn.Linear(ligand_flat_dim + protein_flat_dim, fc_dims[0]),
            nn.BatchNorm1d(fc_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(fc_dims[0], fc_dims[1]),
            nn.BatchNorm1d(fc_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(fc_dims[1], fc_dims[2])
        )
        
    def forward(self, ligand_features, protein_features):
        # Process ligand: [batch, 1024] -> conv layers -> flatten
        lig_x = ligand_features.unsqueeze(1)      # [batch, 1024] -> [batch, 1, 1024]
        lig_x = self.ligand_conv(lig_x)           # Apply conv layers -> [batch, 128, 256]
        lig_x = lig_x.view(lig_x.size(0), -1)     # Flatten -> [batch, 32768]
        
        # Process protein: [batch, 320] -> fully connected layers
        prot_x = self.protein_fc(protein_features)  # [batch, 320] -> [batch, 128]
        
        # Concatenate features
        combined = torch.cat((lig_x, prot_x), dim=1)  # [batch, 32768 + 128] = [batch, 32896]
        
        # Final predictions
        output = self.fc_layers(combined)
        return output

In [5]:
def load_preprocessed_data(data_dir='data/processed_features'):
    """Load preprocessed ligand and protein features"""
    X_lig = np.load(os.path.join(data_dir, f'{output_filename}X_lig.npy')).astype(np.float32)
    X_prot = np.load(os.path.join(data_dir, f'{output_filename}X_prot.npy')).astype(np.float32)
    y = np.load(os.path.join(data_dir, f'{output_filename}y.npy')).astype(np.float32)
    
    print(f"Ligand features shape: {X_lig.shape}")  # Should be [N, 1024]
    print(f"Protein features shape: {X_prot.shape}")  # Should be [N, 320] (ESM embedding)
    
    return X_lig, X_prot, y

In [6]:
def train_model(model, train_loader, val_loader, num_epochs=50, lr=0.01):
    """Train the CNN model"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    best_val_loss = float('inf')
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        
        for batch_idx, (ligand_batch, protein_batch, target_batch) in enumerate(train_loader):
            ligand_batch, protein_batch, target_batch = (
                ligand_batch.to(device), 
                protein_batch.to(device),  # Keep as [batch, 320]
                target_batch.to(device).unsqueeze(1)  # [batch] -> [batch, 1]
            )
            
            optimizer.zero_grad()
            outputs = model(ligand_batch, protein_batch)
            loss = criterion(outputs, target_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for ligand_batch, protein_batch, target_batch in val_loader:
                ligand_batch, protein_batch, target_batch = (
                    ligand_batch.to(device),
                    protein_batch.to(device),
                    target_batch.to(device).unsqueeze(1)
                )
                
                outputs = model(ligand_batch, protein_batch)
                val_loss += criterion(outputs, target_batch).item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_dti_cnn.pth')
        
        if epoch % 10 == 0:
            print(f'Epoch {epoch}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
    
    return model

In [7]:
def evaluate_model(model, test_loader):
    """Evaluate model performance"""
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    model.eval()
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for ligand_batch, protein_batch, target_batch in test_loader:
            ligand_batch, protein_batch, target_batch = (
                ligand_batch.to(device),
                protein_batch.to(device),
                target_batch.to(device)
            )
            
            outputs = model(ligand_batch, protein_batch)
            all_preds.extend(outputs.cpu().numpy())
            all_targets.extend(target_batch.cpu().numpy())
    
    mse = mean_squared_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)
    print(f'Test MSE: {mse:.4f}, R²: {r2:.4f}')

In [7]:
class DTIDataset(TensorDataset):
    """Custom Dataset for Drug-Target Interaction"""
    def __init__(self, X_lig, X_prot, y):
        self.X_lig = X_lig.astype(np.float32)
        self.X_prot = X_prot.astype(np.float32)
        self.y = y.astype(np.float32)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X_lig[idx], self.X_prot[idx], self.y[idx]

In [ ]:
from sklearn.model_selection import train_test_split

# Example usage
if __name__ == "__main__":
    # Load data
    X_lig, X_prot, y = load_preprocessed_data()
    
    # Create datasets
    dataset = TensorDataset(
        torch.tensor(X_lig),      # [N, 1024]
        torch.tensor(X_prot),     # [N, 320] 
        torch.tensor(y)           # [N,] or [N, 1]
    )
    
    # Split data
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
        dataset, [train_size, val_size, test_size]
    )

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    # Initialize model
    model = DTICNN(
        ligand_dim=1024,
        protein_dim=320,  # ESM embedding dimension
        conv_channels=[64, 128],  # Reduced number of conv layers to avoid size issues
        fc_dims=[512, 256, 1],
        dropout_rate=0.3
    )
    
    print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
    print("Ligand conv output shape: [batch, 128, 256]")
    print("Protein FC output shape: [batch, 128]")
    
    # Train model
    trained_model = train_model(model, train_loader, val_loader, num_epochs=150, lr=0.01)
    
    # Evaluate model
    evaluate_model(trained_model, test_loader) # using the test_loader to evaluate the model performance on unseen data
    evaluate_model(trained_model, val_loader) # using the val_loader to evaluate the model performance on validation data

Ligand features shape: (197427, 1024)
Protein features shape: (197427, 320)
Model initialized with 17,031,105 parameters
Ligand conv output shape: [batch, 128, 256]
Protein FC output shape: [batch, 128]
Epoch 0, Train Loss: 1.8585, Val Loss: 1.4063
Epoch 10, Train Loss: 0.7102, Val Loss: 1.1014
Epoch 20, Train Loss: 0.5310, Val Loss: 1.1081
